# Lab 2 — Country Schools: Preprocessing, Anomaly Detection & Clustering

**Dataset:** `Country Schools.csv` (school electrification / solar-kit programme per county in Kenya).

**Objectives**
1. Perform basic data preprocessing (clean, transform, engineer features).
2. Detect anomalous counties using univariate and multivariate methods.
3. Group counties into meaningful clusters and interpret the segments.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
RANDOM_STATE = 42

DATA_PATH = "Country Schools.csv"
print("Pandas", pd.__version__, "| NumPy", np.__version__)

Pandas 3.0.6 | NumPy 2.3.4


## 1. Load and inspect the raw data

In [3]:
df_raw = pd.read_csv(DATA_PATH)

print("Raw shape:", df_raw.shape)
print("\nColumn names:", list(df_raw.columns))
print("\nFirst 5 rows:")
display(df_raw.head())
print("\nLast 6 rows (note the blank rows and TOTAL row):")
display(df_raw.tail(6))
print("\nData types:")
display(df_raw.dtypes.rename("dtype").to_frame())
print("\nMissing values per column:")
display(df_raw.isna().sum().to_frame("missing"))

Raw shape: (50, 8)

Column names: ['SNO', 'TOTALSCHOOLS', 'TotalSchoolsInstalled', 'SchoolsPending', 'PercentageInstalled', 'InstalledLDDs', 'InstalledTDDs', 'FID']

First 5 rows:


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
0,TURKANA,358.00,37.00,321.00,10.34%,"3,744.00",74.00,1
1,HOMA BAY,875.00,552.00,323.00,63.09%,"22,982.00","1,104.00",2
2,WEST POKOT,498.00,242.00,256.00,48.59%,"15,114.00",484.00,3
3,KITUI,"1,238.00","1,200.00",38.00,96.93%,"42,162.00","2,400.00",4
4,TANA RIVER,159.00,143.00,16.00,89.94%,"7,415.00",286.00,5



Last 6 rows (note the blank rows and TOTAL row):


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
44,KAJIADO,395.00,232.00,163.00,58.73%,"8,893.00",464.00,45
45,EMBU,381.00,248.00,133.00,65.09%,"8,438.00",496.00,46
46,SAMBURU,146.00,49.00,97.00,33.56%,"2,891.00",98.00,47
47,TOTAL,"21,729.00","14,629.00","7,100.00",67.32%,"723,618.00","29,258.00",48
48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49
49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50



Data types:


,dtype
SNO,str
TOTALSCHOOLS,float64
TotalSchoolsInstalled,float64
SchoolsPending,float64
PercentageInstalled,str
InstalledLDDs,float64
InstalledTDDs,float64
FID,int64



Missing values per column:


,missing
SNO,2
TOTALSCHOOLS,2
TotalSchoolsInstalled,2
SchoolsPending,2
PercentageInstalled,2
InstalledLDDs,2
InstalledTDDs,2
FID,0


## 2. Data cleaning

In [4]:
df = df_raw.copy()

# (a) drop fully blank rows
df = df.dropna(subset=["TOTALSCHOOLS"]).reset_index(drop=True)
print("After dropping blank rows :", df.shape)

# (b) isolate the TOTAL row for validation, then remove it from analysis
mask_total = df["SNO"].astype(str).str.strip().str.upper() == "TOTAL"
df_total   = df[mask_total].copy()
df         = df[~mask_total].reset_index(drop=True)
print("County rows kept          :", df.shape)
print("TOTAL row saved           :", df_total.shape)

# (c) the column is mislabeled — it holds county names
df = df.rename(columns={"SNO": "County"})
df["County"] = df["County"].astype(str).str.strip().str.title()

# (d) force numeric columns
for c in ["TOTALSCHOOLS","TotalSchoolsInstalled","SchoolsPending",
          "InstalledLDDs","InstalledTDDs","FID"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# (e) convert "10.34%" -> 10.34
df["PercentageInstalled"] = (
    df["PercentageInstalled"].astype(str)
        .str.replace("%", "", regex=False).str.strip()
)
df["PercentageInstalled"] = pd.to_numeric(df["PercentageInstalled"], errors="coerce")

df = df.reset_index(drop=True)

print("\nCleaned shape:", df.shape)
print("Missing cells:", int(df.isna().sum().sum()))
display(df.head())

After dropping blank rows : (48, 8)
County rows kept          : (47, 8)
TOTAL row saved           : (1, 8)

Cleaned shape: (47, 8)
Missing cells: 0


,County,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
0,Turkana,358.00,37.00,321.00,10.34,"3,744.00",74.00,1
1,Homa Bay,875.00,552.00,323.00,63.09,"22,982.00","1,104.00",2
2,West Pokot,498.00,242.00,256.00,48.59,"15,114.00",484.00,3
3,Kitui,"1,238.00","1,200.00",38.00,96.93,"42,162.00","2,400.00",4
4,Tana River,159.00,143.00,16.00,89.94,"7,415.00",286.00,5


In [5]:
print("Data types after cleaning:")
display(df.dtypes.rename("dtype").to_frame())

print("\nNumeric summary:")
display(df.describe().T)

# internal consistency checks
assert (df["TotalSchoolsInstalled"] + df["SchoolsPending"] == df["TOTALSCHOOLS"]).all(), \
       "Installed + Pending must equal Total"
assert np.allclose(df["PercentageInstalled"],
                   df["TotalSchoolsInstalled"] / df["TOTALSCHOOLS"] * 100, atol=0.05)
print("✔ consistency checks passed")

# cross-check against the TOTAL row
print("\nCounty sums vs TOTAL row:")
print("  Total schools     :", df["TOTALSCHOOLS"].sum(), "| TOTAL row:", df_total["TOTALSCHOOLS"].iloc[0])
print("  Installed schools :", df["TotalSchoolsInstalled"].sum(), "| TOTAL row:", df_total["TotalSchoolsInstalled"].iloc[0])

Data types after cleaning:


,dtype
County,str
TOTALSCHOOLS,float64
TotalSchoolsInstalled,float64
SchoolsPending,float64
PercentageInstalled,float64
InstalledLDDs,float64
InstalledTDDs,float64
FID,int64



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
TOTALSCHOOLS,47.00,462.32,254.20,93.00,251.50,413.00,633.50,"1,238.00"
TotalSchoolsInstalled,47.00,311.26,229.00,8.00,156.50,307.00,408.50,"1,200.00"
SchoolsPending,47.00,151.06,113.80,4.00,52.50,130.00,243.00,448.00
PercentageInstalled,47.00,63.92,25.85,4.55,50.30,65.00,84.94,96.93
InstalledLDDs,47.00,"15,396.13","11,479.58",606.00,"6,166.00","13,507.00","22,838.50","44,338.00"
InstalledTDDs,47.00,622.51,458.00,16.00,313.00,614.00,817.00,"2,400.00"
FID,47.00,24.00,13.71,1.00,12.50,24.00,35.50,47.00


✔ consistency checks passed

County sums vs TOTAL row:
  Total schools     : 21729.0 | TOTAL row: 21729.0
  Installed schools : 14629.0 | TOTAL row: 14629.0


### 🧹 Data-quality findings

1. **`SNO` mislabeled** – the column holds county names, not a serial. True ID = `FID`.
2. **`PercentageInstalled` stored as text** – required conversion before any numeric analysis.
3. **`TOTAL` row is an aggregate** – including it would double-count and distort all statistics. It is kept aside for verification only.
4. **`InstalledTDDs = 2 × TotalSchoolsInstalled` for *every* county** – a perfectly deterministic relationship (TDDs are issued at 2 per installed school). This means **`TDDs` adds no information**; `TDDsPerInstalled` will be constant (2.0) and must be dropped from models.
5. **`ElGEYO`** – trailing space and wrong casing fixed by `strip().title()`.